# Tutorial 06 -- Running an IEC DLC sweep

Partial DLC 1.1 coverage at three wind speeds with the v5.0.0 OC3 Tripod r-test deck.

## 1. Verify the OpenFAST binary is installed

In [ ]:
import subprocess
from pathlib import Path

REPO = Path().resolve().parents[1]
binary = REPO / 'tools/openfast/OpenFAST.exe'
if not binary.exists():
    print(f'Binary missing at {binary}')
    print('Download with:')
    print('  curl -L -o tools/openfast/OpenFAST.exe https://github.com/OpenFAST/openfast/releases/download/v5.0.0/OpenFAST.exe')
else:
    out = subprocess.run([str(binary), '-v'], capture_output=True, text=True)
    print(out.stdout[:500])

## 2. Run the DLC 1.1 partial sweep

In [ ]:
# This runs 3 OpenFAST simulations (~5 min each on a modern laptop)
result = subprocess.run(
    ['python', 'scripts/run_dlc11_partial.py', '--tmax', '5', '--speeds', '8', '12', '18'],
    cwd=REPO, capture_output=True, text=True, env={'PYTHONUTF8':'1', **__import__('os').environ},
    timeout=1800,
)
print(result.stdout[-2000:])

## 3. Inspect the output files

In [ ]:
import json
workdirs = sorted((REPO / 'validation/dlc11_partial').glob('*'))
latest = workdirs[-1]
print(f'Latest run: {latest.name}')
summary = json.loads((latest / 'summary.json').read_text())
print(f"Runs: {summary['n_pass']}/{summary['n_runs']} PASS")
for r in summary['runs']:
    print(f"  U = {r['speed_mps']:>5} m/s  rc={r['rc']}  wall={r['wall_seconds']}s  out={r['out_kb']} kB")

## 4. Scaling to full IEC coverage

For full IEC 61400-3 DLC 1.1 coverage:
```bash
python scripts/run_dlc11_partial.py --tmax 600 --speeds 4 6 8 10 11.4 13 15 17 19 21 23 25
```
12 wind speeds × 600 s each ≈ 3.8 hours wall time on a modern laptop. Add multiple seeds per speed for full compliance.

## 5. DLC 6.1 parked extreme wind

In [ ]:
# This run terminates with a physical tower strike because the OC3 Tripod
# controller is in normal-production mode. See scripts/build_dlc61_parked_deck.py
# to build a proper parked-configuration variant.
result = subprocess.run(
    ['python', 'scripts/run_dlc61_parked.py', '--vmax', '50', '--tmax', '3'],
    cwd=REPO, capture_output=True, text=True, env={'PYTHONUTF8':'1', **__import__('os').environ},
    timeout=600,
)
print(result.stdout[-1500:])